In [3]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA, AirQuality
from tsl.data import ImputationDataset, SpatioTemporalDataModule
from einops import rearrange, repeat
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os

from einops.layers.torch import Rearrange
from tsl.nn.layers.norm import Norm

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from pytorch_lightning.loggers import TensorBoardLogger
from tsl.transforms import MaskInput



def find_cliques_size_k(G, k):
    all_cliques = set()
    for clique in nx.find_cliques(G):
        if len(clique) == k:
            all_cliques.add(tuple(sorted(clique)))
        elif len(clique) > k:
            for mini_clique in itertools.combinations(clique, k):
                all_cliques.add(tuple(sorted(mini_clique)))
    return list(all_cliques)




def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(42)

42

In [5]:
add_missing_values(MetrLA(root='./data/metrla'),
                  p_fault=0.0015,
                  p_noise=0.05,
                  min_seq=12,
                  max_seq=12 * 4,
                  seed=9101112)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


MissingValuesMetrLA(length=34272, n_nodes=207, n_channels=1)

In [ ]:
    torch_dataset = ImputationDataset(target=dataset.dataframe(),
                                      mask=dataset.training_mask,
                                      eval_mask=dataset.eval_mask,
                                      covariates=covariates,
                                      transform=MaskInput(),
                                      connectivity=adj,
                                      window=24,
                                      stride=1)

In [7]:
from tsl.ops.imputation import add_missing_values

dataset = add_missing_values(MetrLA(root='./data/metrla'),
                  p_fault=0.0015,
                  p_noise=0.05,
                  min_seq=12,
                  max_seq=12 * 4,
                  seed=9101112)

connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        # normalize_axis=1,
                                        force_symmetric=False,
                                        layout="dense")

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = ImputationDataset(target=dataset.dataframe(),
                                  mask=dataset.training_mask,
                                  eval_mask=dataset.eval_mask,
                                  covariates=covariates,
                                  transform=MaskInput(),
                                  connectivity=connectivity,
                                  window=24,
                                  stride=1)
print(torch_dataset)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


ImputationDataset(n_samples=34249, n_nodes=207, n_channels=1)


In [11]:
torch_dataset.eval_mask.shape

torch.Size([34272, 207, 1])

In [4]:
import math

In [6]:
math.sqrt(23102)

151.9934209102486

In [7]:
math.sqrt(41635)

204.04656331337708

In [35]:
G = nx.from_numpy_array(np.array(torch_dataset.edge_index), create_using =nx.DiGraph)

In [36]:
G.is_directed()

True

In [37]:
nx.reciprocity(G)

0.26666666666666666

In [12]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
    workers=4
)

dm.setup()
print(dm)

{Train dataloader: size=24636}
{Validation dataloader: size=2716}
{Test dataloader: size=6849}
{Predict dataloader: None}


In [ ]:
for i, data in enumerate(dm.train_dataloader()):
    if i == 0:
        print(data)

StaticBatch(
  input=(x=[b=64, t=24, n=207, f=1], u=[b=64, t=24, f=2], mask=[b=64, t=24, n=207, f=1], edge_index=[207, e=207]),
  target=(y=[b=64, t=24, n=207, f=1]),
  has_mask=True,
  transform=[x, y]
)


## Interorder Random Walk

In [20]:
import torch
from torch_cluster import random_walk


def uniform_random_walk(edge_index, nodes, batch_size, num_samples,timesteps,length):
    source, target = edge_index[0], edge_index[1]
    
    num_nodes = len(nodes)
    nodes = nodes.repeat(batch_size * num_samples * timesteps)

    walks, eids = random_walk(row=source,
                              col=target,
                              start=nodes,
                              walk_length=length,
                              return_edge_indices = True)
    
    walks = walks.view(batch_size, num_samples, timesteps, num_nodes, length+1)
    eids = eids.view(batch_size, num_samples, timesteps, num_nodes, length)
    
    return walks, eids


def uniqueness(walk):
    walk_equal = walk.unsqueeze(-1) == walk.unsqueeze(-2)
    # (1 * walk_equal) -- > bool to int such that can use argmax
    walk_equal = (1 * walk_equal).argmax(dim=-1)
    return walk_equal



## Model

In [21]:
def conv_with_norm(input_size, hidden_size, kernel_size, stride, groups):
    modules = nn.Sequential(
        nn.Conv1d(in_channels = input_size,
                     out_channels = hidden_size,
                     kernel_size=kernel_size,
                     stride=stride,
                     # padding='same',
                     groups=groups),
        nn.GELU(),
        # Rearrange('... f t -> ... t f'),
        # nn.LayerNorm(hidden_size),
        # Rearrange('... t f -> ... f t')
        # nn.BatchNorm1d(hidden_size),
    )
    return modules



class DW_large_kernel(nn.Module):
    def __init__(self,
                 input_size,hidden_size,
                 large_kernel,
                 stride, groups):
        super().__init__()

        self.hidden_size = hidden_size
        self.large_kernel = large_kernel
        self.stride = stride


        # only one large kernel
        self.large_conv = conv_with_norm(input_size = input_size, hidden_size = hidden_size,
                                       kernel_size=large_kernel, stride=stride, groups=groups)
    def forward(self, x_emb):
        # causual pad at left side
        x = F.pad(x_emb,
                  pad=((self.large_kernel - 1), 0),
                  mode='constant', value=0)
        out = self.large_conv(x)

        return out

In [22]:
class Backbone_blocks(nn.Module):
    def __init__(self, num_nodes, large_kernel, num_variables, hidden_size, drop=0.1):
        super().__init__()
        self.dw_conv = nn.Sequential(
            DW_large_kernel(num_variables*hidden_size,num_variables*hidden_size,
                              large_kernel,
                              stride=1,
                              groups = num_variables*hidden_size),
            nn.Dropout(p=drop),
            nn.GELU()
            
            
        )
        self.layernorm1 = nn.LayerNorm([num_nodes, hidden_size])
        
        self.pw_con1 = nn.Sequential(
            nn.Conv1d(
            in_channels=num_variables*hidden_size, 
            out_channels=num_variables*hidden_size, 
            kernel_size=1,
            groups=num_variables
            ),
            nn.Dropout(p=drop),
            nn.GELU()
            # Rearrange('... f t -> ... t f'),
            # nn.LayerNorm(num_variables*hidden_size),
            # Rearrange('... t f -> ... f t'),
            
        )
        
        self.pw_con2 = nn.Sequential(
            nn.Conv1d(
            in_channels=num_variables*hidden_size, 
            out_channels=num_variables*hidden_size, 
            kernel_size=1,
            groups=hidden_size
            ),
            nn.Dropout(p=drop),
            nn.GELU()
            # Rearrange('... f t -> ... t f'),
            # nn.LayerNorm(num_variables*hidden_size),
            # Rearrange('... t f -> ... f t'),
            
        )
        self.layernorm2 = nn.LayerNorm([num_nodes, hidden_size])
        # self.layernorm2 = nn.LayerNorm([rw_sample, hidden_size])

    def forward(self, x_emb):
        # x_emb -> [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce]
        batch_size, num_nodes, feature_dim, D, timesteps_reduce = x_emb.shape
        
        # x_emb torch.Size([32, 10, 207, 5, 32, 12])

        x = rearrange(x_emb, 'b n f d t -> (b n) (f d) t').contiguous() 
        x = self.dw_conv(x)

        x = rearrange(x, '(b n) (f d) t -> (b f) t n d', b=batch_size, n=num_nodes, f=feature_dim, d=D)

        x = self.layernorm1(x)
        
        x = rearrange(x, '(b f) t n d -> (b n) (f d) t ',b=batch_size, n=num_nodes, f=feature_dim, d=D)

        x = self.pw_con1(x)
        x = rearrange(x, '(b n) (f d) t -> (b n) (d f) t',
                      b=batch_size, n=num_nodes, f=feature_dim, d=D).contiguous() 

        # x = rearrange(x, '(b s) n f d t -> (b s n) (d f) t',
        #               b=batch_size, s=num_samples)

        x = self.pw_con2(x)
        x = rearrange(x, '(b n) (d f) t -> b d n f t',
                      b=batch_size, d=D,
                      n=num_nodes, f=feature_dim).contiguous() 
        
        # [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce]
        x = rearrange(x, 'b d n f t -> b n f d t').contiguous() 
        
        # residual connection
        out = x + x_emb

        out = rearrange(out, 'b n f d t -> (b f) t n d').contiguous() 

        out = self.layernorm2(out)

        out = rearrange(out, '(b f) t n d -> b t n d f',b=batch_size, n=num_nodes, f=feature_dim, d=D).contiguous() 

        return out

In [23]:
class ModernTCN_rum(pl.LightningModule):
    def __init__(self,
                 input_size,hidden_size,num_nodes,
                 large_kernel,
                 patch_size, patch_stride,
                 windows, horizon,
                 rw_length, rw_sample,
                 num_blocks, downsample_ratio, dropout):
        super().__init__()

        self.rw_length = rw_length
        self.rw_sample = rw_sample
        self.patch_size = patch_size
        self.patch_stride = patch_stride
        self.downsample_ratio = downsample_ratio
        self.windows_patches = windows // patch_stride[0]
        self.rw_length_patches = (rw_length+1) // patch_stride[1]


        # down sampling layers
        self.downsample_layers = nn.ModuleList()

        self.stem = nn.Sequential(
            nn.Conv3d(input_size, hidden_size, kernel_size=patch_size),
            nn.GELU()
        )
        self.layernorm = nn.Sequential(
            # Rearrange('... d t -> ... t d'),
            nn.LayerNorm([num_nodes,hidden_size])
            # Rearrange('... t d -> ... d t')
        )
        # self.downsample_layers.append(stem)

        self.num_blocks = num_blocks

        # if self.num_blocks > 1:
        #     for _ in range(num_blocks-1):
        #         downsample_layer = nn.Conv2d(hidden_size, hidden_size,
        #                                      kernel_size=(downsample_ratio, downsample_ratio),
        #                                      stride=(downsample_ratio, downsample_ratio))
        #         self.downsample_layers.append(downsample_layer)


        # backbones
        self.blocks = nn.ModuleList()
        for block_id in range(num_blocks):
            backbone = Backbone_blocks(num_nodes, large_kernel[block_id],
                                      input_size+4, hidden_size, dropout)
            self.blocks.append(backbone)


        # head
        # else:
        # output_channels = (128 // rw_sample) * rw_sample
        # print("output_channels", output_channels)
        # self.head = nn.Sequential(
        #     Rearrange('b n f -> b f n'),
        #     nn.Conv1d(rw_sample*(input_size+2)*hidden_size*self.windows_patches, output_channels, kernel_size = 1, groups = rw_sample),
        #     nn.GELU(),
        #     nn.Dropout(0.1),
        #     Rearrange('b f n -> b n f'),
        #     nn.LayerNorm(output_channels),
        #     nn.Linear(output_channels, horizon)
        # )

        self.head = nn.Sequential(
            nn.Linear((input_size+4)*hidden_size, 256),
            nn.GELU(),
            nn.Linear(256, horizon)
        )

        self.reset_parameters()

            
        # torch.Size([32, 1, 2, 207, 2, 32, 3])
        # self.head = nn.Sequential(
        #     nn.Linear(rw_sample*(input_size+2)*hidden_size*windows_patches*rw_length_patches,
        #               hidden_size),
        #     nn.SiLU(),
        #     Rearrange('b f n -> b n f'),
        #     nn.Linear(hidden_size, horizon)
        # )


    def reset_parameters(self):
        """
        Simplified initialization using Xavier Gaussian for all weights.
        """
        for name, m in self.named_modules():
            if isinstance(m, nn.Conv1d) or isinstance(m, nn.Conv2d) or isinstance(m, nn.Conv3d):
                # Xavier Gaussian initialization for all convolutional layers
                # if 'dw' in name:
                #     # print(f"Initializing depthwise conv: {name}")
                #     nn.init.normal_(m.weight, 3)
                #     if m.bias is not None:
                #         nn.init.constant_(m.bias, 1)
                # else:
                nn.init.kaiming_normal_(m.weight)
                # Initialize bias if present
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            
            elif isinstance(m, nn.BatchNorm1d):
                # Standard initialization for batch norm
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            
            elif isinstance(m, nn.Linear):
                # Xavier Gaussian for all linear layers
                nn.init.kaiming_normal_(m.weight)
                print(f"Initializing Linear: {name}")
                # Initialize bias
                nn.init.constant_(m.bias, 0)

    def _gather_walk_features(self, features, walks):
        # Move tensors to CPU for processing
        cpu_features = features #--> batch_size, timestep, num_simplices, feature
        cpu_walks = walks #--> batch_size, num_random_walk_sample,timestep,num_nodes,random_walk_length

        # cpu_features torch.Size([64, 12, 1334, 1])
        # cpu_walks torch.Size([64, 5, 12, 207, 6])

        features = rearrange(cpu_features, 'b t n f -> b 1 t n 1 f').contiguous() 
        walks = rearrange(walks, 'b s t n l -> b s t n l 1').contiguous() 

        gathered_features = torch.gather(features.expand(-1, walks.shape[1], -1, -1, walks.shape[4], -1),
                                         3,
                                         walks.expand(-1, -1, -1, -1, -1, cpu_features.shape[3]))

        # Final shape: [batch_size, num_samples, timestep, num_nodes, length, feature_dim]
        # output = rearrange(gathered_features, 'b s t n l f -> b s n l f')
        
        return gathered_features.to(walks.device)

    def single_forward(self, x, u):
        batch_size, num_samples, timesteps, num_nodes, length, _, feature_dim = x.shape

        if u.dim() == 3:
            u = repeat(u, 'b t f -> b s t n l 1 f',
                       s = num_samples, n=num_nodes, l=length)
        x = torch.cat([x, u], -1)

        *_, feature_dim = x.shape

        # x = rearrange(x, 'b s t n l d f -> (b s n f) d t l')
        # _, _, timestep, rw_length = x.shape

        # residual = torch.zeros(1, 1, 1, 1, 1, device=x.device)
        for i in range(self.num_blocks):
            # [batch_size, num_samples, timesteps, num_nodes, length, D, feature_dim] 
            # -> [batch_size * num_samples * num_nodes * feature_dim, D, timesteps, length] 
            if i == 0:
                # x = F.pad(x,
                #           pad=(0, self.patch_size[1]-self.patch_stride[1],
                #                0, self.patch_size[0]-self.patch_stride[0]),
                #                   mode='replicate')
                x = rearrange(x, 'b s t n l d f -> (b n f) d t s l').contiguous() 
                # x = F.pad(x,
                #           pad=(0, 0,
                #                0, 0,
                #                0, self.patch_size[0]-self.patch_stride[0]),
                #                   mode='replicate')

                x = self.stem(x) # --> [batch_size  * num_nodes * feature_dim, D, timesteps_reduced, 1, 1]
                
                x = x.squeeze() # --> [batch_size * num_nodes * feature_dim, D, timesteps_reduced]
                
                # x = self.layernorm(x)
                residual = rearrange(x, '(b n f) d t -> b t n d f',
                          b=batch_size, n=num_nodes, f=feature_dim).contiguous()
            else:
                x = rearrange(residual, 'b t n d f -> (b n f) d t').contiguous()
                
            # else:
            #     if (timestep % self.downsample_ratio) or (rw_length % self.downsample_ratio):
            #         x = F.pad(x,
            #                   pad = (0, self.downsample_ratio - (rw_length % self.downsample_ratio),
            #                          0, self.downsample_ratio - (timestep % self.downsample_ratio)),
            #                   mode='replicate')
            # x = self.downsample_layers[i](x)
            # x = rearrange(x, '(b s n f) d t l -> b s n f d t l',
            #               b=batch_size, s=num_samples, n=num_nodes, f=feature_dim)

            
            
            x = rearrange(x, '(b n f) d t -> b n f d t',
                          b=batch_size, n=num_nodes, f=feature_dim).contiguous() 
            
            x = self.blocks[i](x)

            residual = x + residual

        residual = rearrange(x, 'b t n d f -> (b f) t n d',
                      b=batch_size, n=num_nodes, f=feature_dim).contiguous()

        residual = self.layernorm(residual)

        residual = rearrange(residual, '(b f) t n d -> b t n d f',
                      b=batch_size, n=num_nodes, f=feature_dim).contiguous()
        return residual

    def forward(self, x, edge_index, u):
        # x --> [batch_size, time_steps, num_nodes, features]
        # u --> [batch_size, time_steps, num_nodes, features]

        # print('first')
        # check_tensor(x)
        batch_size, T, num_nodes, features = x.shape
        walks, eids = uniform_random_walk(
            edge_index=edge_index.to('cuda'), 
            nodes=torch.arange(num_nodes).to('cuda'), 
            batch_size = batch_size,
            num_samples=self.rw_sample,
            timesteps = T,
            length=self.rw_length
        ) # -- > [batch_size, num_samples, timesteps, num_nodes, length]
        uniqueness_walk = uniqueness(walks)
        walks, uniqueness_walk = walks.flip(-1), uniqueness_walk.flip(-1)
        # walks, uniqueness_walk = walks, uniqueness_walk
        uniqueness_walk = uniqueness_walk / uniqueness_walk.shape[-1]
        uniqueness_walk = uniqueness_walk * math.pi * 2.0
        uniqueness_walk = torch.cat(
            [
                uniqueness_walk.sin().unsqueeze(-1),
                uniqueness_walk.cos().unsqueeze(-1),
            ],
            dim=-1,
        )

        # Gather features for each node in the walks
        # [batch_size, num_samples, timesteps, num_nodes, length, feature_dim]
        gathered_features = self._gather_walk_features(x, walks)  

        # [batch_size, num_samples, timesteps, num_nodes, length, 1, feature_dim]
        gathered_features = torch.concat([gathered_features, uniqueness_walk], dim = -1)
        x = gathered_features.unsqueeze(-2)
        
        # print(uniqueness_walk[0,0,0,0,:,0,:])
        # torch.Size([32, 1, 12, 207, 6, 1, 3])

        x = self.single_forward(x, u)

        # torch.Size([32, 1, 2, 207, 2, 32, 3])

        x = x[:, -1]
        x = rearrange(x, 'b n d f -> b n (f d)').contiguous() 
        # x = rearrange(x, 'b t n d f -> b n (f d t)').contiguous() 
        pred = self.head(x)

        pred = rearrange(pred, 'b n h -> b h n').contiguous() 

        return pred.unsqueeze(-1)

In [24]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
    'mae': torch_metrics.MaskedMAE(),
    'mse': torch_metrics.MaskedMSE(),
    'mae_step_1': torch_metrics.MaskedMAE(at=0),
   'mae_step_2': torch_metrics.MaskedMAE(at=2),
   'mae_step_3': torch_metrics.MaskedMAE(at=5),
   'mae_step_4': torch_metrics.MaskedMAE(at=11)
}



model = ModernTCN_rum(input_size=1,hidden_size = 64,
                      patch_size = (1,20,6),
                      patch_stride = (1,1,1),
                      large_kernel = [7,5,3],
                      num_nodes=torch_dataset.n_nodes,
                      windows=12, horizon=12,
                      rw_sample=20, rw_length=5,
                      num_blocks=3,downsample_ratio =2, dropout=0.1)


def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}_with_graph"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)


Initializing Linear: head.0
Initializing Linear: head.2


In [25]:
from torch.optim.lr_scheduler import MultiStepLR, CosineAnnealingLR
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 5e-3,
                  # 'weight_decay':1e-3
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[10, 30, 60]}
)

In [26]:
import torch
from pytorch_lightning.callbacks import Callback
import numpy as np

class SimpleGradientMonitor(Callback):
    """Simple callback that monitors gradients to identify vanishing/exploding issues."""
    
    def __init__(self, vanish_threshold=1e-4, explode_threshold=10.0):
        super().__init__()
        self.vanish_threshold = vanish_threshold
        self.explode_threshold = explode_threshold
        
    def on_after_backward(self, trainer, pl_module):
        # Check gradients after backward pass
        for name, param in pl_module.named_parameters():
            if param.grad is not None:
                grad_norm = torch.norm(param.grad, p = 2).item()
                
                # Log gradient issues
                if grad_norm < self.vanish_threshold:
                    print(f"Vanishing gradient in {name}: {grad_norm:.6f}")
                    
                if grad_norm > self.explode_threshold or np.isnan(grad_norm):
                    print(f"Exploding gradient in {name}: {grad_norm:.6f}")
                    
grad_monitor = SimpleGradientMonitor()

In [27]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=5,
        mode='min',
        min_delta = 0.001
    )

trainer = Trainer(
        max_epochs=300,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[0],
        gradient_clip_val=5,
       callbacks=[early_stop_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        precision = '32',
        check_val_every_n_epoch = 5,
        logger=logger    
)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [28]:
trainer.fit(predictor, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | ModernTCN_rum    | 352 K  | train
-----------------------------------------------------------
352 K     Trainable params
0         Non-trainable params
352 K     Total params
1.410     Total estimated model params size (MB)
88        Modules in train mode
0         Modules in eval mode


Training: |                                                                                                   …

Arguments ['edge_weight'] are filtered out. Only args ['edge_index', 'x', 'u'] are forwarded to the model (ModernTCN_rum).


Validation: |                                                                                                 …

Validation: |                                                                                                 …

...

Validation: |                                                                                                 …

Validation: |                                                                                  | 0/? [00:00<?,…

Validation: |                                                                                  | 0/? [00:00<?,…

Validation: |                                                                                  | 0/? [00:00<?,…

Validation: |                                                                                  | 0/? [00:00<?,…

In [29]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at logs/MetrLA/ModernTCN_rum_directed_True_with_graph/version_0/checkpoints/epoch=99-step=15000.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/MetrLA/ModernTCN_rum_directed_True_with_graph/version_0/checkpoints/epoch=99-step=15000.ckpt


Testing: |                                                                                     | 0/? [00:00<?,…

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    2.9944567680358887     │
│         test_mae          │    3.1097006797790527     │
│      test_mae_step_1      │     2.321500778198242     │
│      test_mae_step_2      │    2.7952163219451904     │
│      test_mae_step_3      │    3.1601593494415283     │
│      test_mae_step_4      │     3.534019947052002     │
│         test_mse          │    40.413177490234375     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 3.1097006797790527,
  'test_mae_step_1': 2.321500778198242,
  'test_mae_step_2': 2.7952163219451904,
  'test_mae_step_3': 3.1601593494415283,
  'test_mae_step_4': 3.534019947052002,
  'test_mse': 40.413177490234375,
  'test_loss': 2.9944567680358887}]

In [14]:
co = nn.Conv2d(1, 5, kernel_size = 3)

In [ ]:
co.weight.shape